## 1. Environment Setup
Seed the random generators, import core dependencies, and detect the training device.

In [ ]:
import random

import numpy as np
import torch

from pyhealth.datasets import TUEVDataset
from pyhealth.tasks import EEGEventsTUEV
from pyhealth.datasets.splitter import split_by_sample
from pyhealth.datasets.utils import get_dataloader
from pyhealth.models import TFMTokenizer

SEED = 5
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

## 2. Load TUEV Dataset
Point to the TUEV dataset root and load the dataset.

In [ ]:
dataset = TUEVDataset(
    root='/srv/local/data/TUH/tuh_eeg_events/v2.0.0/edf',  # Update this path
    subset='eval',
    # dev=True
)
dataset.stats()

## 3. Prepare PyHealth Dataset
Set the task for the dataset and convert raw samples into PyHealth format for abnormal EEG classification.

In [ ]:
sample_dataset = dataset.set_task(EEGEventsTUEV(
    resample_rate=200,    # Resample rate
    bandpass_filter=(0.1, 75.0),    # Bandpass filter
    notch_filter=50.0,    # Notch filter
    normalization='95th_percentile'
))

print(f"Total task samples: {len(sample_dataset)}")
print(f"Input schema: {sample_dataset.input_schema}")
print(f"Output schema: {sample_dataset.output_schema}")

# Inspect a sample
sample = sample_dataset[0]
print(f"\nSample keys: {sample.keys()}")
print(f"Signal shape: {sample['signal'].shape}")
print(f"Label: {sample['label']}")

In [ ]:
test_loader = get_dataloader(
    dataset=sample_dataset,
    batch_size=1024,
    shuffle=False,
)
print(f"Test loader size: {len(test_loader)}")

## 4. Initialize TFM-Tokenizer Model

In [ ]:
model = TFMTokenizer(
    dataset=sample_dataset,
    emb_size=64,
    code_book_size=8192,
    trans_freq_encoder_depth=2,
    trans_temporal_encoder_depth=2,
    trans_decoder_depth=8,
    use_classifier=True,
    classifier_depth=4,
)

model = model.to(device)

# tokenizer_checkpoint_path = '[From the TFM-Tokenizer GitHub] - pretrained_weigths/multiple_dataset_settings/Pretrained_tfm_tokenizer_2x2x8/tfm_tokenizer_last.pth'
model.load_pretrained_weights(
    tokenizer_checkpoint_path='/home/jp65/Biosignals_Research/tfm_token_code_for_release/TFM-Tokenizer/pretrained_weigths/multiple_dataset_settings/Pretrained_tfm_tokenizer_2x2x8/tfm_tokenizer_last.pth',
    classifier_checkpoint_path='/home/jp65/PyHealth/TFM_Tokenizer_multiple_finetuned_on_TUEV_1/best_model.pth',
    is_masked_training=False,
    strict=False,
    map_location=device
)

print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

## 5. Test Forward Pass

In [ ]:
batch = next(iter(test_loader))

with torch.no_grad():
    outputs = model(**batch)

print("Output keys:", outputs.keys())
print(f"Loss: {outputs['loss'].item():.4f}")
print(f"Logits shape: {outputs['logit'].shape}")
print(f"Tokens shape: {outputs['tokens'].shape}")
print(f"Embeddings shape: {outputs['embeddings'].shape}")

# 6. Inference on Test Dataloader

In [ ]:
from pyhealth.trainer import Trainer

trainer = Trainer(model=model, device=device,metrics=["accuracy", "balanced_accuracy", "cohen_kappa"],)


In [ ]:
trainer.evaluate(test_loader)